# Training Notebook — DenseNet121 + **RadImageNet** Pretraining

Controlled experiment — identical to the other training notebook in every way except the pretrained weights.
After training, run `xai_pipeline.ipynb` with the saved checkpoint.

```
outputs/
└── radimagenet/seed_42/
    ├── checkpoints/best.pt
    ├── results/
    └── figures/
```


## Cell 0 — Environment setup & data verification

In [1]:
"""Cell 0 — Environment Setup & Data Verification"""
import os, sys, subprocess
from pathlib import Path
import yaml

os.chdir("/workspace")
print(f"  Working directory: {os.getcwd()}")

REQUIREMENTS = Path("requirements.txt")

def install_requirements(req_path):
    if not req_path.is_file():
        print(f"  [WARN] {req_path} not found — skipping pip install.")
        return
    print("Installing packages from requirements.txt ...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(req_path), "-q"],
        capture_output=True, text=True)
    print("  ✔ All packages ready." if r.returncode == 0 else f"  [ERROR] {r.stderr}")
    print("Installing grad-cam ...")
    r2 = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "grad-cam"],
        capture_output=True, text=True)
    print("  ✔ grad-cam ready." if r2.returncode == 0 else f"  [ERROR] {r2.stderr}")

install_requirements(REQUIREMENTS)

CONFIG_PATH = "configs/config.yaml"
with open(CONFIG_PATH, "r") as f:
    _cfg = yaml.safe_load(f)

print("\n" + "=" * 70)
print("Cell 0 — Data Verification")
print("=" * 70)

required = {
    "csv_train":             _cfg["paths"]["csv_train"],
    "csv_val":               _cfg["paths"]["csv_val"],
    "csv_test":              _cfg["paths"]["csv_test"],
    "chexpert_root":         _cfg["paths"]["chexpert_root"],
    "chexlocalize_root":     _cfg["paths"]["chexlocalize_root"],
    "chexlocalize_gt_val":   _cfg["paths"]["chexlocalize_gt_val"],
    "chexlocalize_gt_test":  _cfg["paths"]["chexlocalize_gt_test"],
}

missing = []
for key, path in required.items():
    p = Path(path)
    exists = p.exists()
    marker = "✔" if exists else "✘"
    print(f"  {marker} {key:24s} → {path}")
    if not exists:
        missing.append(key)

if missing:
    print(f"\n  [ERROR] Missing required paths: {missing}")
else:
    print("\n  All required files/directories present.")


  Working directory: /workspace
Installing packages from requirements.txt ...
  ✔ All packages ready.
Installing grad-cam ...
  ✔ grad-cam ready.

Cell 0 — Data Verification
  ✔ csv_train                → /workspace/csvs/train_visualCheXbert.csv
  ✔ csv_val                  → /workspace/csvs/valid.csv
  ✔ csv_test                 → /workspace/csvs/test_labels.csv
  ✔ chexpert_root            → /workspace/CheXpert-v1.0-small
  ✔ chexlocalize_root        → /workspace/chexlocalize
  ✔ chexlocalize_gt_val      → /workspace/chexlocalize/gt_annotations_val.json
  ✔ chexlocalize_gt_test     → /workspace/chexlocalize/gt_annotations_test.json

  All required files/directories present.


## Cell 1 — Imports, seed, config

In [2]:
"""Cell 1 — Imports, Seed Setup, and Configuration Loading"""
import os, json, yaml, random, pickle
from pathlib import Path
from copy import deepcopy
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast
import torchvision
from torchvision import transforms
from torchvision.models import densenet121

from sklearn.metrics import roc_auc_score, average_precision_score

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

def load_config(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

BRANCH_DIR = "radimagenet"

def setup_output_dirs(cfg, seed, branch_dir):
    root = Path(cfg["paths"]["output_root"]) / branch_dir / f"seed_{seed}"
    subdirs = {"root": root,
               "checkpoints": root / "checkpoints",
               "heatmaps":    root / "heatmaps",
               "figures":     root / "figures",
               "results":     root / "results"}
    for p in subdirs.values():
        p.mkdir(parents=True, exist_ok=True)
    return subdirs

CONFIG_PATH = "configs/config.yaml"
CFG  = load_config(CONFIG_PATH)
SEED = CFG["experiment"]["seeds"][0]
set_seed(SEED)

DEVICE    = torch.device(CFG["experiment"]["device"] if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if CFG["experiment"]["amp_dtype"] == "bfloat16" else torch.float16
OUT       = setup_output_dirs(CFG, SEED, BRANCH_DIR)

CLASSES   = CFG["classes"]["selected"]
CSV_NAMES = [c["csv_name"] for c in CLASSES]
CL_NAMES  = [c["cl_name"]  for c in CLASSES]
N_CLASSES = len(CLASSES)

print(f"Branch dir       : {BRANCH_DIR}")
print(f"Device           : {DEVICE}")
print(f"AMP dtype        : {AMP_DTYPE}")
print(f"Seed             : {SEED}")
print(f"Classes          : {N_CLASSES}")
print(f"Output root      : {OUT['root']}")
for c in CLASSES:
    print(f"  - {c['csv_name']:32s} -> {c['cl_name']}")


Branch dir       : radimagenet
Device           : cuda
AMP dtype        : torch.bfloat16
Seed             : 42
Classes          : 6
Output root      : /workspace/outputs/radimagenet/seed_42
  - Enlarged Cardiomediastinum       -> Enlarged Cardiomediastinum
  - Cardiomegaly                     -> Cardiomegaly
  - Lung Opacity                     -> Lung Opacity
  - Edema                            -> Edema
  - Atelectasis                      -> Atelectasis
  - Pleural Effusion                 -> Pleural Effusion


## Cell 1b — Data Pipeline Overview

Three CSVs are loaded directly:
- `train_visualCheXbert.csv` — VisualCheXbert-relabeled training set (clean labels)
- `valid.csv` — CheXpert official validation (3-radiologist gold standard)
- `test_labels.csv` — CheXpert official test (5/8-radiologist gold standard)

The CheXLocalize test set (668 frontal images) is the same physical images as `test_labels.csv` — they are shared between classification eval (here) and XAI/localization eval in `xai_pipeline.ipynb`.


In [3]:
"""Cell 1b — Data Pipeline Sanity Check"""
import pandas as pd
from pathlib import Path

print("=" * 65)
print("Cell 1b — Data Pipeline Sanity Check")
print("=" * 65)

paths = {
    "Train": CFG["paths"]["csv_train"],
    "Val":   CFG["paths"]["csv_val"],
    "Test":  CFG["paths"]["csv_test"],
}

for name, p in paths.items():
    p = Path(p)
    if not p.is_file():
        print(f"  ✘ {name:6s}: MISSING — {p}")
        continue
    df = pd.read_csv(p)
    cols = list(df.columns)
    n_rows = len(df)
    has_frontal_col = "Frontal/Lateral" in cols
    if has_frontal_col:
        n_frontal = (df["Frontal/Lateral"].astype(str).str.lower() == "frontal").sum()
    else:
        n_frontal = df["Path"].astype(str).str.lower().str.contains("frontal").sum() if "Path" in cols else 0
    print(f"  ✔ {name:6s}: {n_rows:>7,} rows  |  {n_frontal:>6,} frontal  |  cols={len(cols)}")

print("\nCell 1b complete.")


Cell 1b — Data Pipeline Sanity Check
  ✔ Train : 223,414 rows  |  191,027 frontal  |  cols=19
  ✔ Val   :     234 rows  |     202 frontal  |  cols=19
  ✔ Test  :     668 rows  |     518 frontal  |  cols=15

Cell 1b complete.


## Cell 2 — Load CSVs, build label matrix, load CheXLocalize annotations

Steps performed:
1. Load three user-provided CSVs.
2. Apply label mapping `{"": 0, "1.0": 1, "0.0": 0, "-1.0": 0}` (no uncertainty policy needed).
3. Resolve image paths across CheXpert and CheXLocalize roots.
4. Filter frontal-only.
5. Verify patient-level non-overlap between train, val, and test partitions.
6. Load CheXLocalize ground-truth polygons for downstream XAI evaluation.


In [4]:
"""Cell 2 — Data Loading"""
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

LABEL_MAP = {
    "":     0,
    "1.0":  1, "1":  1,
    "0.0":  0, "0":  0,
    "-1.0": 0, "-1": 0,
    "nan":  0, "NaN": 0,
}

def map_label_value(v):
    if pd.isna(v):
        return 0
    return LABEL_MAP.get(str(v).strip(), 0)


def shift_visualchexbert_columns(df):
    if "Support Devices" not in df.columns:
        return df
    if df.columns[-1] != "Support Devices":
        return df
    cols = list(df.columns)
    cols.remove("Support Devices")
    insert_at = 5 if len(cols) >= 5 else len(cols)
    cols.insert(insert_at, "Support Devices")
    return df[cols]


def resolve_image_path(rel_path, split, chexpert_root, chexlocalize_root):
    if not isinstance(rel_path, str) or not rel_path:
        return ""
    rel = rel_path.replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]
            break
    parts = rel.split("/", 1)
    has_split_prefix = len(parts) > 1 and parts[0] in ("train", "valid", "val", "test")
    sub_rel = parts[1] if has_split_prefix else rel

    if split == "train":
        p = Path(chexpert_root) / "train" / sub_rel
        if p.is_file(): return str(p)
        p2 = Path(chexpert_root) / rel
        if p2.is_file(): return str(p2)
    elif split == "val":
        p = Path(chexpert_root) / "valid" / sub_rel
        if p.is_file(): return str(p)
        p2 = Path(chexpert_root) / rel
        if p2.is_file(): return str(p2)
    elif split == "test":
        for candidate in [
            Path(chexlocalize_root) / "test" / "test" / sub_rel,
            Path(chexlocalize_root) / "test" / sub_rel,
            Path(chexlocalize_root) / rel,
        ]:
            if candidate.is_file(): return str(candidate)
    return ""


def load_split(csv_path, classes, split, chexpert_root, chexlocalize_root,
               frontal_only=True):
    df = pd.read_csv(csv_path)
    if "visualchexbert" in str(csv_path).lower():
        df = shift_visualchexbert_columns(df)

    if frontal_only and "Frontal/Lateral" in df.columns:
        df = df[df["Frontal/Lateral"].astype(str).str.strip().str.lower() == "frontal"].copy()
    elif frontal_only and "Path" in df.columns:
        df = df[df["Path"].astype(str).str.lower().str.contains("frontal")].copy()
    df = df.reset_index(drop=True)

    for c in classes:
        col = c["csv_name"]
        if col not in df.columns:
            raise KeyError(f"{csv_path}: missing column {col!r}. Available: {list(df.columns)}")
        df[col] = df[col].apply(map_label_value).astype(np.int8)

    def _unified_key(path):
        if pd.isna(path): return ""
        parts = str(path).replace("\\", "/").split("/")
        if len(parts) >= 3:
            return (f"{parts[-3]}_{parts[-2]}_"
                    f"{parts[-1].replace('.jpg','').replace('.dcm','')}")
        return ""

    def _patient_id(path):
        m = re.search(r"(patient\d+)", str(path))
        return m.group(1) if m else "unknown"

    df["unified_key"] = df["Path"].apply(_unified_key)
    df["patient_id"]  = df["Path"].apply(_patient_id)
    df["img_path"]    = df["Path"].apply(
        lambda p: resolve_image_path(p, split, chexpert_root, chexlocalize_root))

    n_before = len(df)
    df = df[df["img_path"] != ""].reset_index(drop=True)
    if len(df) < n_before:
        print(f"  [{split}] dropped {n_before - len(df):,} rows with unresolved paths")

    keep_cols = ["unified_key", "patient_id", "img_path"] + [c["csv_name"] for c in classes]
    return df[keep_cols]


def class_distribution(df, classes):
    n_total = len(df)
    rows = []
    for c in classes:
        col = c["csv_name"]
        n_pos = int((df[col] == 1).sum())
        n_neg = int((df[col] == 0).sum())
        rows.append({
            "class":     col,
            "positive":  n_pos,
            "negative":  n_neg,
            "pos_ratio": round(n_pos / n_total, 4) if n_total > 0 else 0.0,
        })
    return pd.DataFrame(rows)


def polygon_to_bbox(polygon):
    arr = np.asarray(polygon, dtype=np.float32)
    return (float(arr[:, 0].min()), float(arr[:, 1].min()),
            float(arr[:, 0].max()), float(arr[:, 1].max()))


def load_chexlocalize_gt(json_path, cl_class_names):
    with open(json_path, "r") as f:
        raw = json.load(f)
    cl_set = set(cl_class_names)
    out = {}
    for img_key, entry in raw.items():
        img_size = entry.get("img_size", None)
        boxes = {}
        for cls_name, polys in entry.items():
            if cls_name == "img_size" or cls_name not in cl_set or not polys:
                continue
            boxes[cls_name] = [polygon_to_bbox(p) for p in polys]
        out[img_key] = {"img_size": img_size, "boxes": boxes}
    return out


print("Loading CSVs ...")

CHEXPERT_ROOT     = CFG["paths"]["chexpert_root"]
CHEXLOCALIZE_ROOT = CFG["paths"]["chexlocalize_root"]

df_train = load_split(CFG["paths"]["csv_train"], CLASSES, "train",
                      CHEXPERT_ROOT, CHEXLOCALIZE_ROOT)
df_val   = load_split(CFG["paths"]["csv_val"],   CLASSES, "val",
                      CHEXPERT_ROOT, CHEXLOCALIZE_ROOT)
df_test  = load_split(CFG["paths"]["csv_test"],  CLASSES, "test",
                      CHEXPERT_ROOT, CHEXLOCALIZE_ROOT)

print(f"  Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

print("\nPatient-level overlap check:")
pat_train = set(df_train["patient_id"].unique())
pat_val   = set(df_val["patient_id"].unique())
pat_test  = set(df_test["patient_id"].unique())
overlap_tv  = pat_train & pat_val
overlap_tt  = pat_train & pat_test
overlap_vt  = pat_val   & pat_test
print(f"  Train ∩ Val : {len(overlap_tv):,} patients")
print(f"  Train ∩ Test: {len(overlap_tt):,} patients")
print(f"  Val   ∩ Test: {len(overlap_vt):,} patients")
if overlap_tv or overlap_tt or overlap_vt:
    print("  [WARN] Patient-level overlap detected.")
else:
    print("  ✔ No patient-level overlap.")

print("\nClass distribution — Train:")
dist_train = class_distribution(df_train, CLASSES)
print(dist_train.to_string(index=False))

print("\nClass distribution — Val:")
dist_val = class_distribution(df_val, CLASSES)
print(dist_val.to_string(index=False))

print("\nClass distribution — Test:")
dist_test = class_distribution(df_test, CLASSES)
print(dist_test.to_string(index=False))

dist_train.to_csv(OUT["results"] / "class_dist_train.csv", index=False)
dist_val.to_csv(OUT["results"]   / "class_dist_val.csv",   index=False)
dist_test.to_csv(OUT["results"]  / "class_dist_test.csv",  index=False)

print("\nLoading CheXLocalize annotations ...")
gt_val  = load_chexlocalize_gt(CFG["paths"]["chexlocalize_gt_val"],  CL_NAMES)
gt_test = load_chexlocalize_gt(CFG["paths"]["chexlocalize_gt_test"], CL_NAMES)
print(f"  CheXLocalize val:  {len(gt_val)} images")
print(f"  CheXLocalize test: {len(gt_test)} images")

def _cl_counts(gt):
    counts = defaultdict(int)
    for v in gt.values():
        for cn in v["boxes"].keys():
            counts[cn] += 1
    return pd.Series(counts).reindex(CL_NAMES).fillna(0).astype(int)

cl_summary = pd.DataFrame({"val": _cl_counts(gt_val), "test": _cl_counts(gt_test)})
print("\nCheXLocalize positives per class (val / test):")
print(cl_summary)
cl_summary.to_csv(OUT["results"] / "chexlocalize_class_counts.csv")


Loading CSVs ...
  Train: 191,027  |  Val: 202  |  Test: 518

Patient-level overlap check:
  Train ∩ Val : 0 patients
  Train ∩ Test: 0 patients
  Val   ∩ Test: 0 patients
  ✔ No patient-level overlap.

Class distribution — Train:
                     class  positive  negative  pos_ratio
Enlarged Cardiomediastinum    131225     59802     0.6869
              Cardiomegaly    111420     79607     0.5833
              Lung Opacity    137342     53685     0.7190
                     Edema     88580    102447     0.4637
               Atelectasis    112217     78810     0.5874
          Pleural Effusion     93189     97838     0.4878

Class distribution — Val:
                     class  positive  negative  pos_ratio
Enlarged Cardiomediastinum       105        97     0.5198
              Cardiomegaly        66       136     0.3267
              Lung Opacity       117        85     0.5792
                     Edema        42       160     0.2079
               Atelectasis        75       127

## Cell 3 — Dataset, CLAHE preprocessing & DataLoaders

In [5]:
"""Cell 3 — Dataset, Transforms, DataLoaders"""

class CLAHE:
    def __init__(self, clip_limit=2.0, tile_grid_size=8):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit,
                                     tileGridSize=(tile_grid_size, tile_grid_size))
    def __call__(self, img_uint8):
        if img_uint8.ndim == 3:
            img_uint8 = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        return self.clahe.apply(img_uint8)


class CheXpertDataset(Dataset):
    def __init__(self, df, classes, image_size, mean, std,
                 augment, clahe_clip, clahe_grid, rotation_deg):
        self.df = df.reset_index(drop=True)
        self.csv_names = [c["csv_name"] for c in classes]
        self.image_size = image_size
        self.augment = augment
        self.clahe = CLAHE(clip_limit=clahe_clip, tile_grid_size=clahe_grid)
        self.rotation_deg = rotation_deg
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std  = torch.tensor(std).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.array(Image.open(path).convert("L"))
        return self.clahe(img)

    def _augment_resize(self, img):
        if self.augment and self.rotation_deg > 0:
            angle = float(np.random.uniform(-self.rotation_deg, self.rotation_deg))
            h, w = img.shape
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        return cv2.resize(img, (self.image_size, self.image_size),
                          interpolation=cv2.INTER_AREA)

    def _to_tensor(self, img):
        x = torch.from_numpy(img).float().div(255.0)
        x = x.unsqueeze(0).repeat(3, 1, 1)
        return (x - self.mean) / self.std

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load(row["img_path"])
        img = self._augment_resize(img)
        x = self._to_tensor(img)
        labels = np.array([row[c] for c in self.csv_names], dtype=np.float32)
        return x, torch.from_numpy(labels), row["unified_key"]


def compute_pos_weight(df, classes):
    pw = []
    for c in classes:
        col = df[c["csv_name"]]
        n_pos = int((col == 1).sum())
        n_neg = int((col == 0).sum())
        pw.append(max(n_neg, 1) / max(n_pos, 1))
    return torch.tensor(pw, dtype=torch.float32)


def build_dataloaders(cfg, df_train, df_val, df_test, classes):
    d = cfg["data"]
    common = dict(image_size=d["image_size"], mean=d["norm_mean"], std=d["norm_std"],
                  clahe_clip=d["clahe_clip"], clahe_grid=d["clahe_grid"],
                  rotation_deg=d["rotation_deg"])
    ds_train = CheXpertDataset(df_train, classes, augment=True,  **common)
    ds_val   = CheXpertDataset(df_val,   classes, augment=False, **common)
    ds_test  = CheXpertDataset(df_test,  classes, augment=False, **common)
    nw = cfg["experiment"]["num_workers"]
    pm = cfg["experiment"]["pin_memory"]
    dl_train = DataLoader(ds_train, batch_size=d["batch_size"], shuffle=True,
                          num_workers=nw, pin_memory=pm, drop_last=True,
                          persistent_workers=(nw > 0))
    dl_val   = DataLoader(ds_val,   batch_size=d["batch_size"], shuffle=False,
                          num_workers=nw, pin_memory=pm,
                          persistent_workers=(nw > 0))
    dl_test  = DataLoader(ds_test,  batch_size=d["batch_size"], shuffle=False,
                          num_workers=nw, pin_memory=pm,
                          persistent_workers=(nw > 0))
    return dl_train, dl_val, dl_test


print("\nBuilding DataLoaders ...")
dl_train, dl_val, dl_test = build_dataloaders(CFG, df_train, df_val, df_test, CLASSES)
pos_weight = compute_pos_weight(df_train, CLASSES)
print(f"  Train batches: {len(dl_train)}  |  Val batches: {len(dl_val)}  |  Test batches: {len(dl_test)}")
print(f"  Batch size   : {CFG['data']['batch_size']}")
print(f"  pos_weight per class:")
for c, w in zip(CSV_NAMES, pos_weight.tolist()):
    print(f"    {c:32s}: {w:.3f}")



Building DataLoaders ...
  Train batches: 1492  |  Val batches: 2  |  Test batches: 5
  Batch size   : 128
  pos_weight per class:
    Enlarged Cardiomediastinum      : 0.456
    Cardiomegaly                    : 0.714
    Lung Opacity                    : 0.391
    Edema                           : 1.157
    Atelectasis                     : 0.702
    Pleural Effusion                : 1.050


## Cell 4 — Model architecture (shared)

In [6]:
"""Cell 4 — DenseNet121CXR architecture"""
class DenseNet121CXR(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        base = densenet121(weights=None)
        self.features   = base.features
        in_feats        = base.classifier.in_features
        self.classifier = nn.Linear(in_feats, n_classes)

    def forward(self, x):
        feat = self.features(x)
        feat = F.relu(feat, inplace=True)
        feat = F.adaptive_avg_pool2d(feat, (1, 1))
        feat = torch.flatten(feat, 1)
        return self.classifier(feat)


## Cell 4b — Load RadImageNet weights

In [7]:
"""Cell 4b — DenseNet121 + RadImageNet weights"""

def load_radimagenet_weights(model, weights_path, verbose=True):
    sd = torch.load(weights_path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    if isinstance(sd, dict) and "model_state" in sd:
        sd = sd["model_state"]
    remapped = {}
    for k, v in sd.items():
        kk = k
        if kk.startswith("module."):
            kk = kk[len("module."):]
        if kk.startswith("model."):
            kk = kk[len("model."):]
        if kk.startswith("features."):
            remapped[kk] = v
        elif kk.startswith("denseblock") or kk.startswith("conv0") or kk.startswith("norm0") \
             or kk.startswith("pool0") or kk.startswith("transition"):
            remapped[f"features.{kk}"] = v
    missing, unexpected = model.load_state_dict(remapped, strict=False)
    if verbose:
        bm = [k for k in missing if not k.startswith("classifier.")]
        print(f"  Source        : {weights_path}")
        print(f"  Loaded keys   : {len(remapped)}")
        print(f"  Backbone miss : {len(bm)} (expected 0)")
        print(f"  Classifier    : fresh (expected 2 missing)")
        print(f"  Unexpected    : {len(unexpected)}")
        if bm:
            print("  ! WARNING — missing backbone keys (first 5):", bm[:5])
        if unexpected:
            print("  ! WARNING — unexpected keys (first 5):", unexpected[:5])
    return list(missing), list(unexpected)

def build_model(cfg, n_classes, device):
    model = DenseNet121CXR(n_classes=n_classes)
    load_radimagenet_weights(model, cfg["paths"]["radimagenet_weights"], verbose=True)
    return model.to(device)

print("Building DenseNet121 + RadImageNet ...")
model = build_model(CFG, N_CLASSES, DEVICE)
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")
with torch.no_grad():
    dummy = torch.randn(2, 3, CFG["data"]["image_size"], CFG["data"]["image_size"], device=DEVICE)
    out   = model(dummy)
print(f"  Output shape: {tuple(out.shape)}  (expect (2, {N_CLASSES}))")


Building DenseNet121 + RadImageNet ...
  Source        : /workspace/weights/DenseNet121.pt
  Loaded keys   : 0
  Backbone miss : 604 (expected 0)
  Classifier    : fresh (expected 2 missing)
  Unexpected    : 0
  ! WARNING — missing backbone keys (first 5): ['features.conv0.weight', 'features.norm0.weight', 'features.norm0.bias', 'features.norm0.running_mean', 'features.norm0.running_var']
  Params: 6,960,006
  Output shape: (2, 6)  (expect (2, 6))


## Cell 5 — Training loop (BCE + pos_weight)

In [8]:
"""Cell 5 — Training (BCE + pos_weight)"""
BRANCH_NAME = "RadImageNet"

from torch.cuda.amp import GradScaler
import time

print("=" * 70)
print(f"Cell 5 — Training DenseNet121 ({BRANCH_NAME})")
print("=" * 70)

TR = CFG["training"]
EPOCHS = int(TR["epochs"])
LR     = float(TR["lr"])
WD     = float(TR["weight_decay"])
PAT    = int(TR["early_stopping_patience"])

print(f"  Loss type   : BCE + pos_weight")
print(f"  Epochs      : {EPOCHS}")
print(f"  Optimizer   : Adam(lr={LR}, weight_decay={WD})")
print(f"  Scheduler   : CosineAnnealingLR(T_max={EPOCHS})")
print(f"  Early stop  : patience={PAT}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

amp_enabled = (DEVICE.type == "cuda")
amp_dtype   = AMP_DTYPE if amp_enabled else torch.float32


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    all_logits, all_labels = [], []
    total_loss, n_batches = 0.0, 0
    for xb, yb, *_ in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb_dev = yb.to(DEVICE, non_blocking=True)
        with autocast(dtype=amp_dtype, enabled=amp_enabled):
            logits = model(xb)
            if criterion is not None:
                loss_val = criterion(logits, yb_dev)
                total_loss += float(loss_val.item())
                n_batches += 1
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(yb.numpy())

    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    probs  = 1.0 / (1.0 + np.exp(-logits))

    aurocs, aps, supports = {}, {}, {}
    for i, cls in enumerate(CSV_NAMES):
        y = labels[:, i]; p = probs[:, i]
        if np.unique(y).size < 2:
            aurocs[cls] = float("nan"); aps[cls] = float("nan")
        else:
            aurocs[cls] = float(roc_auc_score(y, p))
            aps[cls]    = float(average_precision_score(y, p))
        supports[cls] = int(len(y))

    return {
        "loss":             total_loss / max(n_batches, 1),
        "auroc_per_class":  aurocs,
        "ap_per_class":     aps,
        "support":          supports,
        "macro_auroc":      float(np.nanmean(list(aurocs.values()))),
        "mean_ap":          float(np.nanmean(list(aps.values()))),
    }


ckpt_dir = OUT["checkpoints"]; ckpt_dir.mkdir(parents=True, exist_ok=True)
best_macro, no_improve = -1.0, 0
history = []
t_global = time.time()

print(f"\n  Training for {EPOCHS} epochs ...")
print("=" * 70)

for ep in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running, n_batches = 0.0, 0
    pbar = tqdm(dl_train, desc=f"Epoch {ep}/{EPOCHS}", leave=False)

    for xb, yb, *_ in pbar:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=amp_dtype, enabled=amp_enabled):
            logits = model(xb)
            loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        bl = float(loss.item())
        running += bl; n_batches += 1
        pbar.set_postfix({"loss": f"{bl:.4f}"})

    scheduler.step()
    train_loss = running / max(n_batches, 1)
    val_m = evaluate(model, dl_val, criterion=criterion)

    print(f"\n  Epoch {ep:>2d}/{EPOCHS}  "
          f"trainLoss={train_loss:.4f}  "
          f"valLoss={val_m['loss']:.4f}  "
          f"macroAUROC={val_m['macro_auroc']:.4f}  "
          f"meanAP={val_m['mean_ap']:.4f}  "
          f"time={time.time()-t0:.0f}s")
    for cn, v in val_m["auroc_per_class"].items():
        print(f"      {cn:32s} AUROC={v:.4f}  AP={val_m['ap_per_class'][cn]:.4f}")

    history.append({
        "epoch": ep, "train_loss": train_loss, "val_loss": val_m["loss"],
        "val_macro_auroc": val_m["macro_auroc"], "val_mean_ap": val_m["mean_ap"],
        "val_per_class_auroc": val_m["auroc_per_class"],
    })

    if val_m["macro_auroc"] > best_macro:
        best_macro = val_m["macro_auroc"]; no_improve = 0
        torch.save({"model_state":  model.state_dict(),
                    "epoch":        ep,
                    "val_metrics":  val_m,
                    "macro_auroc":  best_macro,
                    "loss_type":    "bce_pos_weight",
                    "branch":       BRANCH_NAME},
                   ckpt_dir / "best.pt")
        print(f"      ✔ new best macroAUROC={best_macro:.4f}  → checkpoint saved")
    else:
        no_improve += 1
        print(f"      no improvement ({no_improve}/{PAT})")
        if no_improve >= PAT:
            print(f"\n  Early stopping at epoch {ep}")
            break

print(f"\n  Total training time: {(time.time()-t_global)/60:.1f} min")
with open(OUT["results"] / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\n" + "=" * 70)
print("Final classification evaluation on TEST set (one-shot)")
print("  → uses test_labels.csv (5/8 radiologists)")
print("  → images are CheXLocalize test set (same 668 images used later for XAI)")
print("=" * 70)

best = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model_state"]); model.to(DEVICE).eval()
print(f"  Loaded best.pt  (epoch {best['epoch']}, val macroAUROC={best['val_metrics']['macro_auroc']:.4f})")

test_m = evaluate(model, dl_test, criterion=criterion)
print(f"\n  TEST  loss={test_m['loss']:.4f}  "
      f"macroAUROC={test_m['macro_auroc']:.4f}  meanAP={test_m['mean_ap']:.4f}\n")
print(f"  {'Class':<34s}{'AUROC':>10s}{'AP':>10s}{'#used':>10s}")
print("  " + "-" * 64)
for cls in CSV_NAMES:
    print(f"  {cls:<34s}{test_m['auroc_per_class'][cls]:>10.4f}"
          f"{test_m['ap_per_class'][cls]:>10.4f}{test_m['support'][cls]:>10d}")

with open(OUT["results"] / "classification_metrics.json", "w") as f:
    json.dump({"loss_type": "bce_pos_weight",
               "best_epoch": best["epoch"],
               "val":  best["val_metrics"],
               "test": test_m}, f, indent=2)
print(f"\n  ✔ saved {OUT['results'] / 'classification_metrics.json'}")


Cell 5 — Training DenseNet121 (RadImageNet)
  Loss type   : BCE + pos_weight
  Epochs      : 15
  Optimizer   : Adam(lr=0.0001, weight_decay=1e-05)
  Scheduler   : CosineAnnealingLR(T_max=15)
  Early stop  : patience=4

  Training for 15 epochs ...


Epoch 1/15:   0%|          | 0/1492 [00:00<?, ?it/s]


  Epoch  1/15  trainLoss=0.3732  valLoss=0.4174  macroAUROC=0.8446  meanAP=0.7450  time=476s
      Enlarged Cardiomediastinum       AUROC=0.8415  AP=0.7975
      Cardiomegaly                     AUROC=0.7566  AP=0.5539
      Lung Opacity                     AUROC=0.8790  AP=0.9077
      Edema                            AUROC=0.8456  AP=0.6146
      Atelectasis                      AUROC=0.8519  AP=0.7507
      Pleural Effusion                 AUROC=0.8929  AP=0.8454
      ✔ new best macroAUROC=0.8446  → checkpoint saved


Epoch 2/15:   0%|          | 0/1492 [00:00<?, ?it/s]


  Epoch  2/15  trainLoss=0.3422  valLoss=0.4363  macroAUROC=0.8549  meanAP=0.7656  time=462s
      Enlarged Cardiomediastinum       AUROC=0.8440  AP=0.8194
      Cardiomegaly                     AUROC=0.7742  AP=0.6072
      Lung Opacity                     AUROC=0.8852  AP=0.9139
      Edema                            AUROC=0.8671  AP=0.6511
      Atelectasis                      AUROC=0.8632  AP=0.7701
      Pleural Effusion                 AUROC=0.8955  AP=0.8320
      ✔ new best macroAUROC=0.8549  → checkpoint saved


...


  Epoch  7/15  trainLoss=0.3014  valLoss=0.4288  macroAUROC=0.8584  meanAP=0.7671  time=333s
      Enlarged Cardiomediastinum       AUROC=0.8225  AP=0.7899
      Cardiomegaly                     AUROC=0.7768  AP=0.6088
      Lung Opacity                     AUROC=0.8908  AP=0.9141
      Edema                            AUROC=0.8840  AP=0.6738
      Atelectasis                      AUROC=0.8702  AP=0.7777
      Pleural Effusion                 AUROC=0.9060  AP=0.8383
      no improvement (2/4)


Epoch 8/15:   0%|          | 0/1492 [00:00<?, ?it/s]


  Epoch  8/15  trainLoss=0.2901  valLoss=0.4624  macroAUROC=0.8516  meanAP=0.7515  time=338s
      Enlarged Cardiomediastinum       AUROC=0.8101  AP=0.7749
      Cardiomegaly                     AUROC=0.7573  AP=0.5652
      Lung Opacity                     AUROC=0.8897  AP=0.9139
      Edema                            AUROC=0.8830  AP=0.6452
      Atelectasis                      AUROC=0.8650  AP=0.7613
      Pleural Effusion                 AUROC=0.9046  AP=0.8484
      no improvement (3/4)


Epoch 9/15:   0%|          | 0/1492 [00:00<?, ?it/s]


  Epoch  9/15  trainLoss=0.2751  valLoss=0.4776  macroAUROC=0.8473  meanAP=0.7478  time=344s
      Enlarged Cardiomediastinum       AUROC=0.8078  AP=0.7807
      Cardiomegaly                     AUROC=0.7589  AP=0.5692
      Lung Opacity                     AUROC=0.8903  AP=0.9157
      Edema                            AUROC=0.8766  AP=0.6663
      Atelectasis                      AUROC=0.8544  AP=0.7235
      Pleural Effusion                 AUROC=0.8958  AP=0.8312
      no improvement (4/4)

  Early stopping at epoch 9

  Total training time: 59.8 min

Final classification evaluation on TEST set (one-shot)
  → uses test_labels.csv (5/8 radiologists)
  → images are CheXLocalize test set (same 668 images used later for XAI)
  Loaded best.pt  (epoch 5, val macroAUROC=0.8617)

  TEST  loss=0.3785  macroAUROC=0.8786  meanAP=0.7440

  Class                                  AUROC        AP     #used
  ----------------------------------------------------------------
  Enlarged Cardiomediast